# Generate Example Images from Microscopy Data

This notebook generates example images from microscopy experiments with flexible configuration options.

## What this notebook does:
1. Loads a plate layout CSV with experimental conditions
2. Matches it to image metadata
3. Selects specific fields from wells
4. Calculates intensity rescaling limits per group
5. Saves rescaled images for specified channels
6. Optionally creates segmentation overlays

## Before you start:
- Prepare your plate layout CSV file
- Know the path to your images (and segmentations if needed)
- Decide which conditions/groups you want to visualize

## Import Libraries

In [ ]:
# Standard library imports
import os
import glob
import warnings
from pathlib import Path

# Data manipulation
import numpy as np
import pandas as pd

# Image processing
import mahotas as mh
from PIL import Image
from aicsimageio import AICSImage, readers

# Visualization
import matplotlib
import matplotlib.pyplot as plt

# Progress bars
from tqdm.auto import tqdm

# Optional: Illumination correction
try:
    from blimp.preprocessing.illumination_correction import IlluminationCorrection
    ILLUM_CORRECTION_AVAILABLE = True
except ImportError:
    ILLUM_CORRECTION_AVAILABLE = False
    warnings.warn("blimp package not found. Illumination correction will not be available.")

print("✓ All libraries imported successfully")

---
# Configuration Section
**Edit the parameters below to match your experiment**

In [ ]:
# ============================================================================
# FILE PATHS
# ============================================================================

# Path to your plate layout CSV file (contains experimental metadata)
plate_layout_file = "/srv/scratch/z3532965/src/blana/Scott/20260626_POLR2A_heterogeneity/METADATA/20260626_single_cell_POLR2A_plate_layout.csv"

# Path to directory containing images
image_dir = "/srv/scratch/berrylab/z3532965/systems_Ti2/20260626_POLR2A_heterogeneity/20260629_134305_894/OME-TIFF-MIP/"

# Path to directory containing segmentation label images (optional, set to None if not using)
segmentation_dir = "/srv/scratch/berrylab/z3532965/systems_Ti2/20260626_POLR2A_heterogeneity/20260629_134305_894/SEGMENTATION/"  # or None
#segmentation_dir = None

# Path to illumination correction file (optional, set to None if not using)
#illumination_correction_file = "/path/to/illumination_correction.pkl"  # or None
illumination_correction_file = "/srv/scratch/berrylab/z3532965/systems_Ti2/20260626_POLR2A_heterogeneity/20260629_134305_894/illumination_correction.pkl"

# Output directory for saved images
output_dir = "/srv/scratch/berrylab/z3532965/systems_Ti2/20260626_POLR2A_heterogeneity/20260629_134305_894/EXAMPLES"

# Regex pattern to extract well name from filename (optional)
# Example: For "Well_A01_Channel..." use r'Well[_]?([A-Z]\d{2})'
# Leave as None to use the well column from metadata as-is
well_extraction_pattern = r'Well([A-Z]\d{2})_Channel'

In [ ]:
# Load plate layout
plate_layout = None
if os.path.exists(plate_layout_file):
    plate_layout = pd.read_csv(plate_layout_file, dtype=str)
    plate_layout = plate_layout.where(pd.notnull(plate_layout), "None")
    print(f"✓ Plate layout: {len(plate_layout)} rows\n")
    
    for col in plate_layout.columns:
        unique_values = plate_layout[col].unique()
        n_unique = len(unique_values)
        
        if n_unique <= 8 and all(len(str(v)) <= 20 for v in unique_values):
            print(f"  {col}: {list(unique_values)}")
        else:
            print(f"  {col}: {n_unique} unique values")
    
    print(f"\n{plate_layout.head()}\n")
else:
    print(f"⚠ Plate layout file not found: {plate_layout_file}\n")

# Load image metadata
image_metadata = None
if os.path.exists(image_dir):
    image_metadata_files = glob.glob(os.path.join(image_dir, "*.csv"))
    
    if image_metadata_files:
        image_metadata = pd.concat((pd.read_csv(f, dtype=str) for f in image_metadata_files), ignore_index=True)
        print(f"✓ Image metadata: {len(image_metadata)} rows from {len(image_metadata_files)} file(s)\n")
        
        for col in image_metadata.columns:
            unique_values = image_metadata[col].unique()
            n_unique = len(unique_values)
            
            if n_unique <= 8 and all(len(str(v)) <= 20 for v in unique_values):
                print(f"  {col}: {list(unique_values)}")
            else:
                print(f"  {col}: {n_unique} unique values")
        
        print(f"\n{image_metadata.head()}")
    else:
        print(f"⚠ No CSV files found in: {image_dir}")
else:
    print(f"⚠ Image directory not found: {image_dir}")

## Load and Preview Data

Load plate layout and image metadata files. These will be used throughout the notebook.

In [ ]:
# ============================================================================
# COLUMN MAPPING
# ============================================================================

# Column name in plate layout CSV that contains well identifiers (e.g., "WellName", "well_name")
plate_layout_well_column = "well"

# Column name in image metadata CSV that contains well identifiers
image_metadata_well_column = "well"

# Column name in image metadata CSV that contains the image filename
image_metadata_filename_column = "filename_ome_tiff"  # Common alternatives: "filename_ome_tiff", "Filename"

# Column name in image metadata that contains field ID
image_metadata_field_column = "field_id"

In [ ]:
# ============================================================================
# IMAGE SELECTION
# ============================================================================

# Which field ID to select for generating examples (integer)
# Example: If you have fields 1-9, and want field 4, set this to 4
selected_field_id = '4'

# Filter query for additional selection criteria (optional, set to None for no filtering)
# Example: "Treatment == 'Control'" or "TimePoint <= 24"
# additional_filter = None
additional_filter = (
    "PRIMARY_RAT != 'None' "
    "and PRIMARY_RABBIT != 'None' "
    "and not (SECONDARY_488 == 'None' and SECONDARY_568 == 'None' and SECONDARY_647 == 'None')"
)

In [ ]:
# ============================================================================
# GROUPING VARIABLES
# ============================================================================

# Columns from plate layout used to identify unique experimental conditions
# These will be used in the output filename
# Example: ["Construct", "Treatment", "Treatment_Duration", "Treatment_Concentration"]
grouping_variables = ["CELLS", "PRIMARY_RAT", "PRIMARY_RABBIT", "SECONDARY_488", "SECONDARY_568", "SECONDARY_647"]

# Columns used to group images for calculating intensity rescaling limits
# (May differ from grouping_variables if different conditions should share the same scaling)
# Example: ["Primary", "Secondary"] to scale all replicates of the same primary antibody together
rescaling_grouping_variables = ["PRIMARY_RAT", "PRIMARY_RABBIT", "SECONDARY_488", "SECONDARY_568", "SECONDARY_647"]

# Select only one representative well per unique combination of grouping variables
# If True, only the first well from each condition will be processed
# If False, all replicate wells will be processed
select_one_well_per_condition = True  # or False to process all replicates

# List of control wells to save with each rescaling group (optional, set to None or empty list)
# These wells will be saved multiple times, once with each rescaling group's intensity limits
# Example: ["A01", "B01", "H12"] for negative control wells
# Set to None or [] to skip control well processing
control_wells = ['E03','G03','I03','K03','M03','E16','G16','I16','K16','M16','E17','E18','G17','G18','I17','I18','K17','K18','M17','M18','F18','J18']  # or ["A01", "B01"]

In [ ]:
# ============================================================================
# CHANNEL SETTINGS
# ============================================================================

# Dictionary mapping channel indices to colormaps
# Key: channel index (0-based), Value: matplotlib colormap name
# Common colormaps: 'magma', 'viridis', 'gray', 'green', 'red', 'cyan'
# See visualization at end of notebook for full colormap list
channels_to_save = {
    # 0: 'gray',
    1: 'gray',
    2: 'gray',
    4: 'gray',
    5: 'gray',
}

# Intensity rescaling parameters
lower_rescale_value = 120        # Lower intensity limit
upper_quantile = 0.995           # Quantile for upper intensity limit (0-1)

In [ ]:
# ============================================================================
# SEGMENTATION OVERLAY SETTINGS (optional)
# ============================================================================

# Path to segmentation images (set to None if not generating overlays)
# Segmentation filenames should match the image filenames exactly
# Example: If image is "image001.ome.tiff", segmentation should be "image001.ome.tiff"
segmentation_path = segmentation_dir 

# If segmentation_path is provided, specify overlay details:

# Channel index in the segmentation image to use for overlays (0-based)
segmentation_channel = 0

# Border thickness for overlay outlines
overlay_thin_borders = True  # True for thin borders, False for thicker borders

---
# Code Section
**You shouldn't need to edit anything below this line**

## Validate Configuration

Check that all required parameters are set correctly.

In [ ]:
# Validate configuration
errors = []

# Check that data was loaded
if plate_layout is None:
    errors.append(f"Plate layout file not found: {plate_layout_file}")

if image_metadata is None:
    errors.append(f"No image metadata found in: {image_dir}")

# Check file paths
if not os.path.exists(image_dir):
    errors.append(f"Image directory not found: {image_dir}")

if segmentation_dir is not None and not os.path.exists(segmentation_dir):
    errors.append(f"Segmentation directory not found: {segmentation_dir}")

if illumination_correction_file is not None and not os.path.exists(illumination_correction_file):
    errors.append(f"Illumination correction file not found: {illumination_correction_file}")

# Check illumination correction availability
if illumination_correction_file is not None and not ILLUM_CORRECTION_AVAILABLE:
    errors.append("Illumination correction file specified but blimp package not available")

# Check channel settings
if not channels_to_save:
    errors.append("No channels specified in channels_to_save")

# Check segmentation settings
if segmentation_path is not None and not os.path.exists(segmentation_path):
    errors.append(f"Segmentation path not found: {segmentation_path}")

if errors:
    print("❌ Configuration errors found:")
    for error in errors:
        print(f"  • {error}")
    raise ValueError("Please fix configuration errors above")
else:
    print("✓ Configuration validated successfully")
    print(f"\nSettings summary:")
    print(f"  • Plate layout: {len(plate_layout)} wells loaded")
    print(f"  • Image metadata: {len(image_metadata)} images loaded")
    print(f"  • Field ID: {selected_field_id}")
    print(f"  • Channels: {list(channels_to_save.keys())}")
    print(f"  • Grouping by: {grouping_variables}")
    print(f"  • Rescaling grouped by: {rescaling_grouping_variables}")
    print(f"  • Segmentation overlays: {'Yes' if segmentation_path is not None else 'No'}")

# Extract well name from filename if pattern is provided
if well_extraction_pattern is not None:
    if image_metadata_filename_column in image_metadata.columns:
        image_metadata['well_name_extracted'] = image_metadata[image_metadata_filename_column].str.extract(well_extraction_pattern)
        print(f"\n✓ Extracted well names from filenames using pattern: {well_extraction_pattern}")
        print(f"  Example: {image_metadata[image_metadata_filename_column].iloc[0]} → {image_metadata['well_name_extracted'].iloc[0]}")
    else:
        raise ValueError(f"Cannot extract well names: column '{image_metadata_filename_column}' not found in image metadata")

## Load Illumination Correction (if specified)

In [ ]:
illumination_correction = None

if illumination_correction_file is not None:
    print(f"Loading illumination correction from: {illumination_correction_file}")
    illumination_correction = IlluminationCorrection(from_file=illumination_correction_file)
    print("✓ Illumination correction loaded")
else:
    print("ℹ Skipping illumination correction (no file specified)")

## Validate and Merge Metadata

Validate column names and merge plate layout with image metadata.

In [ ]:
# Validate plate layout has required column
if plate_layout_well_column not in plate_layout.columns:
    raise ValueError(f"Column '{plate_layout_well_column}' not found in plate layout. "
                     f"Available columns: {list(plate_layout.columns)}")

# Validate grouping variables exist
missing_vars = [var for var in grouping_variables if var not in plate_layout.columns]
if missing_vars:
    raise ValueError(f"Grouping variables not found in plate layout: {missing_vars}")

missing_rescale_vars = [var for var in rescaling_grouping_variables if var not in plate_layout.columns]
if missing_rescale_vars:
    raise ValueError(f"Rescaling grouping variables not found in plate layout: {missing_rescale_vars}")

# Validate image metadata has required columns
required_cols = [image_metadata_filename_column, image_metadata_field_column]
# Use extracted well_name if pattern was provided, otherwise use the well column from metadata
if well_extraction_pattern is not None:
    if 'well_name_extracted' not in image_metadata.columns:
        raise ValueError("Well extraction pattern provided but well_name_extracted column not found")
    merge_column = 'well_name_extracted'
else:
    required_cols.append(image_metadata_well_column)
    merge_column = image_metadata_well_column

missing_cols = [col for col in required_cols if col not in image_metadata.columns]
if missing_cols:
    raise ValueError(f"Required columns not found in image metadata: {missing_cols}. "
                     f"Available columns: {list(image_metadata.columns)}")

In [ ]:
# Merge plate layout with image metadata
print("Merging plate layout with image metadata...")
image_metadata_annotated = image_metadata.merge(
    plate_layout,
    left_on=merge_column,
    right_on=plate_layout_well_column,
    how='inner'
)

print(f"✓ Merged data: {len(image_metadata_annotated)} images with annotations")

if len(image_metadata_annotated) == 0:
    raise ValueError("No matches found between plate layout and image metadata. "
                     f"Check that well names match between files. "
                     f"Merging on: image_metadata['{merge_column}'] = plate_layout['{plate_layout_well_column}']")

## Filter Images

Select images based on field ID and any additional filters.

In [ ]:
# Filter by field ID
print(f"Filtering for field ID: {selected_field_id}")
filtered_images = image_metadata_annotated[
    image_metadata_annotated[image_metadata_field_column] == selected_field_id
].copy()

print(f"✓ After field filter: {len(filtered_images)} images")

# Apply additional filter if specified
if additional_filter is not None:
    print(f"Applying additional filter: {additional_filter}")
    filtered_images = filtered_images.query(additional_filter)
    print(f"✓ After additional filter: {len(filtered_images)} images")

if len(filtered_images) == 0:
    raise ValueError("No images remaining after filtering. Check your filter criteria.")

# Select one well per condition if requested
if select_one_well_per_condition:
    print(f"\nSelecting one representative well per condition...")
    # Group by the grouping variables and take the first well from each group
    filtered_images = filtered_images.groupby(grouping_variables, as_index=False).first()
    print(f"✓ Selected {len(filtered_images)} representative wells (one per condition)")

# Display summary of conditions
print("\nConditions to be processed:")
condition_summary = filtered_images.groupby(grouping_variables).size()
print(condition_summary)

In [ ]:
filtered_images.columns

## Prepare Output Filenames

Generate output filenames based on grouping variables.

In [ ]:
# Create grouping identifier for filenames
print("Generating output filenames...")

# Build filename from grouping variables
if len(grouping_variables) == 1:
    filtered_images['condition_id'] = filtered_images[grouping_variables[0]].astype(str)
else:
    # Join multiple columns element-wise
    filtered_images['condition_id'] = filtered_images[grouping_variables[0]].astype(str)
    for var in grouping_variables[1:]:
        filtered_images['condition_id'] = filtered_images['condition_id'] + '_' + filtered_images[var].astype(str)

# Add well and field to filename
filtered_images['base_filename'] = (
    filtered_images['condition_id'] + '_' +
    filtered_images[image_metadata_well_column].astype(str) + '_' +
    'Field' + filtered_images[image_metadata_field_column].astype(str)
)

# Create rescaling group identifier
if len(rescaling_grouping_variables) == 1:
    filtered_images['rescaling_group'] = filtered_images[rescaling_grouping_variables[0]].astype(str)
else:
    # Join multiple columns element-wise
    filtered_images['rescaling_group'] = filtered_images[rescaling_grouping_variables[0]].astype(str)
    for var in rescaling_grouping_variables[1:]:
        filtered_images['rescaling_group'] = filtered_images['rescaling_group'] + '_' + filtered_images[var].astype(str)

print(f"✓ Generated filenames for {len(filtered_images)} images")
print(f"\nExample filenames:")
print(filtered_images['base_filename'].head(3).tolist())

## Calculate Intensity Rescaling Limits

Calculate the upper rescaling limit for each rescaling group based on the specified quantile.

In [ ]:
print("Calculating intensity rescaling limits...")
print(f"Using quantile: {upper_quantile}")

upper_rescale_values = {}

# Group images by rescaling group
rescaling_groups = filtered_images.groupby('rescaling_group')
print(f"Processing {len(rescaling_groups)} rescaling group(s)...\n")

for group_name, group_data in tqdm(rescaling_groups, desc="Rescaling groups"):
    group_upper_limits = {ch: [] for ch in channels_to_save.keys()}
    
    # Calculate upper limit for each channel
    for _, row in group_data.iterrows():
        image_path = Path(image_dir) / row[image_metadata_filename_column]
        
        try:
            # Load image
            aics_image = AICSImage(image_path, reader=readers.ome_tiff_reader.OmeTiffReader)
            
            # Apply illumination correction if available
            if illumination_correction is not None:
                aics_image = illumination_correction.correct(aics_image)
            
            # Calculate quantile for each channel
            for channel_idx in channels_to_save.keys():
                channel_array = aics_image.get_image_data('YX', Z=0, C=channel_idx, T=0)
                upper_limit = np.quantile(channel_array, upper_quantile)
                group_upper_limits[channel_idx].append(upper_limit)
        
        except Exception as e:
            warnings.warn(f"Error processing {image_path}: {e}")
            continue
    
    # Store maximum upper limit for each channel in this group
    for channel_idx in channels_to_save.keys():
        if group_upper_limits[channel_idx]:
            if group_name not in upper_rescale_values:
                upper_rescale_values[group_name] = {}
            upper_rescale_values[group_name][channel_idx] = max(group_upper_limits[channel_idx])

print(f"\n✓ Calculated rescaling limits for {len(upper_rescale_values)} group(s)")

# Create dataframe for easier merging
rescale_records = []
for group, channels in upper_rescale_values.items():
    for channel, value in channels.items():
        rescale_records.append({
            'rescaling_group': group,
            'channel': channel,
            'upper_rescale_value': value
        })

rescale_df = pd.DataFrame(rescale_records)
print("\nRescaling limits:")
print(rescale_df)

## Helper Function for Segmentation Overlays

In [ ]:
def create_overlay_image(base_image, label_image, thin=True):
    """
    Create an overlay of segmentation outlines on a base image.
    
    Parameters:
    -----------
    base_image : PIL.Image
        Base image to overlay on (should be RGBA)
    label_image : numpy.ndarray
        Label image with integer labels for each object
    thin : bool
        If True, use thin borders. If False, use dilated borders.
    
    Returns:
    --------
    PIL.Image
        Image with overlay applied
    """
    label_image = np.array(label_image)
    
    # Generate outlines
    if thin:
        outlines = mh.labeled.borders(label_image) * 255
    else:
        outlines = mh.morph.dilate(mh.labeled.borders(label_image)) * 255
    
    # Create overlay
    overlay = Image.fromarray(np.uint8(outlines))
    outlines_transparent = Image.new(
        mode='RGBA', size=outlines.shape[::-1], color=(0, 0, 0, 0)
    )
    
    # Convert base to RGBA if needed
    base_image = base_image.convert("RGBA")
    
    # Paste base and overlay
    outlines_transparent.paste(base_image, (0, 0))
    outlines_transparent.paste(overlay, (0, 0), mask=overlay)
    
    return outlines_transparent

print("✓ Overlay helper function defined")

## Generate and Save Images

Process each image, apply rescaling, and save outputs.

In [ ]:
# Create output directories
output_path = Path(output_dir)
for channel_idx in channels_to_save.keys():
    channel_dir = output_path / f"Channel_{channel_idx}"
    channel_dir.mkdir(parents=True, exist_ok=True)
    
    if segmentation_path is not None:
        overlay_dir = output_path / f"Channel_{channel_idx}_Overlay"
        overlay_dir.mkdir(parents=True, exist_ok=True)

print(f"✓ Created output directories in: {output_path}")
print(f"\nProcessing {len(filtered_images)} images...")

In [ ]:
# Process and save images
processing_errors = []

for idx, row in tqdm(filtered_images.iterrows(), total=len(filtered_images), desc="Processing images"):
    image_path = Path(image_dir) / row[image_metadata_filename_column]
    base_filename = row['base_filename']
    rescaling_group = row['rescaling_group']
    
    try:
        # Load image
        aics_image = AICSImage(image_path, reader=readers.ome_tiff_reader.OmeTiffReader)
        
        # Apply illumination correction if available
        if illumination_correction is not None:
            aics_image = illumination_correction.correct(aics_image)
        
        # Process each channel
        for channel_idx, colormap_name in channels_to_save.items():
            # Extract channel data
            channel_array = aics_image.get_image_data('YX', Z=0, C=channel_idx, T=0)
            
            # Get rescaling limits for this group and channel
            upper_limit = upper_rescale_values[rescaling_group][channel_idx]
            
            # Rescale intensities
            channel_rescaled = (channel_array.astype(float) - lower_rescale_value) / upper_limit
            channel_rescaled = channel_rescaled.clip(0, 1)
            
            # Apply colormap and convert to RGB
            colormap = matplotlib.colormaps[colormap_name]
            rgb_image = Image.fromarray(colormap(channel_rescaled, bytes=True)).convert('RGB')
            
            # Save base image
            output_filename = f"{base_filename}_Ch{channel_idx}.png"
            output_file_path = output_path / f"Channel_{channel_idx}" / output_filename
            rgb_image.save(str(output_file_path), quality=85, subsampling=0)
            
            # Generate segmentation overlay if requested
            if segmentation_path is not None:
                # Use same filename as image
                seg_filename = row[image_metadata_filename_column]
                seg_file_path = Path(segmentation_path) / seg_filename
                
                if seg_file_path.exists():
                    seg_image = AICSImage(seg_file_path, reader=readers.ome_tiff_reader.OmeTiffReader)
                    seg_array = seg_image.get_image_data('YX', Z=0, C=segmentation_channel, T=0)
                    overlay_image = create_overlay_image(
                        rgb_image,
                        seg_array,
                        thin=overlay_thin_borders
                    )
                    
                    # Save overlay image
                    overlay_filename = f"{base_filename}_Ch{channel_idx}_overlay.png"
                    overlay_file_path = output_path / f"Channel_{channel_idx}_Overlay" / overlay_filename
                    overlay_image.save(str(overlay_file_path), quality=85, subsampling=0)
                else:
                    warnings.warn(f"Segmentation file not found: {seg_file_path}")
    
    except Exception as e:
        error_msg = f"Error processing {image_path}: {str(e)}"
        processing_errors.append(error_msg)
        warnings.warn(error_msg)

print(f"\n✓ Processing complete!")
print(f"  • Successfully processed: {len(filtered_images) - len(processing_errors)} images")
if processing_errors:
    print(f"  • Errors encountered: {len(processing_errors)}")
    print("\nError details:")
    for error in processing_errors[:5]:  # Show first 5 errors
        print(f"  • {error}")
    if len(processing_errors) > 5:
        print(f"  ... and {len(processing_errors) - 5} more errors")

## Process Control Wells (Optional)

If control wells are specified, save them with each rescaling group's intensity limits.

In [ ]:
if control_wells and len(control_wells) > 0:
    print(f"\n{'='*70}")
    print("PROCESSING CONTROL WELLS")
    print(f"{'='*70}")
    print(f"Control wells: {control_wells}")
    print(f"Will be saved with {len(upper_rescale_values)} different rescaling group(s)\n")
    
    # Filter for control wells
    control_images = image_metadata_annotated[
        (image_metadata_annotated[image_metadata_well_column].isin(control_wells)) &
        (image_metadata_annotated[image_metadata_field_column] == selected_field_id)
    ].copy()
    
    if len(control_images) == 0:
        print("⚠ Warning: No control well images found matching the specified wells and field ID")
    else:
        print(f"Found {len(control_images)} control well image(s)")
        
        # Create control output directories
        for channel_idx in channels_to_save.keys():
            control_dir = output_path / f"Control_Images" / f"Channel_{channel_idx}"
            control_dir.mkdir(parents=True, exist_ok=True)
            
            if segmentation_path is not None:
                control_overlay_dir = output_path / f"Control_Images" / f"Channel_{channel_idx}_Overlay"
                control_overlay_dir.mkdir(parents=True, exist_ok=True)
        
        control_errors = []
        
        # Process each control image with each rescaling group
        for rescale_group_name, rescale_limits in tqdm(
            upper_rescale_values.items(), 
            desc="Processing control wells with different rescaling"
        ):
            for idx, row in control_images.iterrows():
                image_path = Path(image_dir) / row[image_metadata_filename_column]
                well_name = row[image_metadata_well_column]
                field_id = row[image_metadata_field_column]
                
                try:
                    # Load image
                    aics_image = AICSImage(image_path, reader=readers.ome_tiff_reader.OmeTiffReader)
                    
                    # Apply illumination correction if available
                    if illumination_correction is not None:
                        aics_image = illumination_correction.correct(aics_image)
                    
                    # Process each channel
                    for channel_idx, colormap_name in channels_to_save.items():
                        # Extract channel data
                        channel_array = aics_image.get_image_data('YX', Z=0, C=channel_idx, T=0)
                        
                        # Get rescaling limits for this rescaling group and channel
                        upper_limit = rescale_limits[channel_idx]
                        
                        # Rescale intensities
                        channel_rescaled = (channel_array.astype(float) - lower_rescale_value) / upper_limit
                        channel_rescaled = channel_rescaled.clip(0, 1)
                        
                        # Apply colormap and convert to RGB
                        colormap = matplotlib.colormaps[colormap_name]
                        rgb_image = Image.fromarray(colormap(channel_rescaled, bytes=True)).convert('RGB')
                        
                        # Save base image with rescaling group in filename
                        control_filename = f"Control_{well_name}_Field{field_id}_RescaledAs_{rescale_group_name}_Ch{channel_idx}.png"
                        control_output_path = output_path / f"Control_Images" / f"Channel_{channel_idx}" / control_filename
                        rgb_image.save(str(control_output_path), quality=85, subsampling=0)
                        
                        # Generate segmentation overlay if requested
                        if segmentation_path is not None:
                            seg_filename = row[image_metadata_filename_column]
                            seg_file_path = Path(segmentation_path) / seg_filename
                            
                            if seg_file_path.exists():
                                seg_image = AICSImage(seg_file_path, reader=readers.ome_tiff_reader.OmeTiffReader)
                                seg_array = seg_image.get_image_data('YX', Z=0, C=segmentation_channel, T=0)
                                overlay_image = create_overlay_image(
                                    rgb_image,
                                    seg_array,
                                    thin=overlay_thin_borders
                                )
                                
                                # Save overlay image
                                control_overlay_filename = f"Control_{well_name}_Field{field_id}_RescaledAs_{rescale_group_name}_Ch{channel_idx}_overlay.png"
                                control_overlay_path = output_path / f"Control_Images" / f"Channel_{channel_idx}_Overlay" / control_overlay_filename
                                overlay_image.save(str(control_overlay_path), quality=85, subsampling=0)
                
                except Exception as e:
                    error_msg = f"Error processing control {image_path} with rescaling {rescale_group_name}: {str(e)}"
                    control_errors.append(error_msg)
                    warnings.warn(error_msg)
        
        print(f"\n✓ Control well processing complete!")
        print(f"  • Total control images generated: {len(control_images) * len(upper_rescale_values) * len(channels_to_save)}")
        if control_errors:
            print(f"  • Errors encountered: {len(control_errors)}")
else:
    print("\nℹ Skipping control well processing (no control wells specified)")

## Save Colormap References

Generate and save colormap scale bars for each channel.

In [ ]:
print("Generating colormap scale bars...")

for channel_idx, colormap_name in channels_to_save.items():
    # Create a gradient from 0 to 1
    gradient = np.linspace(0, 1, 256)
    gradient = np.vstack([gradient] * 50)  # Make it taller for better visibility
    
    # Apply colormap
    colormap = matplotlib.colormaps[colormap_name]
    colormap_image = colormap(gradient)
    
    # Convert to RGB image (0-255 range)
    rgb_array = (colormap_image[:, :, :3] * 255).astype(np.uint8)
    
    # Create PIL image and save
    colormap_pil = Image.fromarray(rgb_array)
    colormap_filename = f"colormap_Ch{channel_idx}_{colormap_name}.png"
    colormap_path = output_path / colormap_filename
    colormap_pil.save(str(colormap_path), quality=85, subsampling=0)
    
    print(f"  • Saved colormap for Channel {channel_idx} ({colormap_name})")

print(f"✓ Colormap scale bars saved in: {output_path}")

## Summary

Display final summary of generated images.

In [ ]:
print("=" * 70)
print("PROCESSING COMPLETE")
print("=" * 70)
print(f"\nOutput directory: {output_path}")
print(f"\nGenerated images:")
for channel_idx in channels_to_save.keys():
    channel_dir = output_path / f"Channel_{channel_idx}"
    num_files = len(list(channel_dir.glob("*.png")))
    print(f"  • Channel {channel_idx}: {num_files} images")
    
    if segmentation_path is not None:
        overlay_dir = output_path / f"Channel_{channel_idx}_Overlay"
        num_overlay_files = len(list(overlay_dir.glob("*.png")))
        print(f"    └─ Overlays: {num_overlay_files} images")

if control_wells and len(control_wells) > 0:
    print(f"\nControl images:")
    for channel_idx in channels_to_save.keys():
        control_dir = output_path / f"Control_Images" / f"Channel_{channel_idx}"
        if control_dir.exists():
            num_control_files = len(list(control_dir.glob("*.png")))
            print(f"  • Channel {channel_idx}: {num_control_files} images")
            
            if segmentation_path is not None:
                control_overlay_dir = output_path / f"Control_Images" / f"Channel_{channel_idx}_Overlay"
                if control_overlay_dir.exists():
                    num_control_overlay = len(list(control_overlay_dir.glob("*.png")))
                    print(f"    └─ Overlays: {num_control_overlay} images")

print(f"\nColormap scale bars: {len(channels_to_save)} saved")

print(f"\nConditions processed:")
for condition, count in filtered_images.groupby(grouping_variables).size().items():
    print(f"  • {condition}: {count} image(s)")

---
## Reference: Available Matplotlib Colormaps

Run the cell below to visualize all available colormaps for channel selection.

In [ ]:
def plot_colormaps(cmap_list, title):
    """Plot a set of colormaps as gradients."""
    gradient = np.linspace(0, 1, 256)
    gradient = np.vstack((gradient, gradient))
    
    nrows = len(cmap_list)
    fig, axes = plt.subplots(nrows=nrows, figsize=(8, 0.3 * nrows))
    fig.suptitle(title, fontsize=14, y=0.98)
    
    if nrows == 1:
        axes = [axes]
    
    for ax, cmap_name in zip(axes, cmap_list):
        ax.imshow(gradient, aspect='auto', cmap=cmap_name)
        ax.text(-0.01, 0.5, cmap_name, va='center', ha='right', 
                fontsize=10, transform=ax.transAxes)
        ax.set_axis_off()
    
    plt.tight_layout()
    return fig

# Common colormaps for microscopy
print("Common colormaps for microscopy:\n")

fig1 = plot_colormaps(
    ['gray', 'Greys', 'viridis', 'plasma', 'inferno', 'magma', 'cividis'],
    'Perceptually Uniform Colormaps'
)
plt.show()

fig2 = plot_colormaps(
    ['Greens', 'Reds', 'Blues', 'Purples', 'Oranges'],
    'Sequential Color Colormaps'
)
plt.show()

print("\nFor a full list of colormaps, visit:")
print("https://matplotlib.org/stable/tutorials/colors/colormaps.html")